In [ ]:
import os
import cv2
import numpy as np
import random

# === CONFIGURATION ===
input_root = '/content/drive/MyDrive/Datasets/UTFVP_ROI'
output_root = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
os.makedirs(output_root, exist_ok=True)

# === AUGMENTATION FUNCTIONS ===
def luminance_transform(image):
    factor = np.random.uniform(0.6, 1.5)  # random brightness
    hsv = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    hsv = cv2.cvtColor(hsv, cv2.COLOR_BGR2HSV)
    hsv[..., 2] = np.clip(hsv[..., 2] * factor, 0, 255)
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def add_gaussian_noise(image):
    std = np.random.uniform(10, 30)
    noise = np.random.normal(0, std, image.shape).astype(np.float32)
    return np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def random_affine(image):
    dx, dy = random.randint(-4, 4), random.randint(-4, 4)
    rows, cols = image.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(image, M, (cols, rows), borderMode=cv2.BORDER_REFLECT)

def random_rotation(image):
    angle = np.random.uniform(-10, 10)
    rows, cols = image.shape
    M = cv2.getRotationMatrix2D((cols / 2, rows / 2), angle, 1)
    return cv2.warpAffine(image, M, (cols, rows), borderMode=cv2.BORDER_REFLECT)

def random_blur(image):
    k = random.choice([3, 5])
    return cv2.GaussianBlur(image, (k, k), 0)

def add_occlusion(image, occ_size=(10, 10)):
    img = image.copy()
    h, w = img.shape
    y = np.random.randint(0, h - occ_size[0])
    x = np.random.randint(0, w - occ_size[1])
    img[y:y + occ_size[0], x:x + occ_size[1]] = 0
    return img

# === MAIN AUGMENTATION LOOP ===
for folder in sorted(os.listdir(input_root)):
    input_sub = os.path.join(input_root, folder)
    output_sub = os.path.join(output_root, folder)
    os.makedirs(output_sub, exist_ok=True)

    all_images = sorted([f for f in os.listdir(input_sub) if f.endswith('.png')])

    for fname in all_images:
        img_path = os.path.join(input_sub, fname)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        base = os.path.splitext(fname)[0]

        # === Save Original ===
        cv2.imwrite(os.path.join(output_sub, fname), image)

        for aug_id in range(1, 4):  # Generate 3 randomized augmentations
            img_aug = image.copy()

            # Randomized sequence of all augmentations
            img_aug = luminance_transform(img_aug)
            img_aug = random_rotation(img_aug)
            img_aug = random_affine(img_aug)
            img_aug = add_gaussian_noise(img_aug)
            img_aug = random_blur(img_aug)
            img_aug = add_occlusion(img_aug)

            aug_name = f"{base}_{aug_id}_Augmented.png"
            cv2.imwrite(os.path.join(output_sub, aug_name), img_aug)

print("✅ All original + 3 randomized full-augmentation images saved per input.")
